# Load Packages

In [16]:
import os
import pandas as pd
from patsy import dmatrix
# import matplotlib.pyplot as plt

# Parameters

In [ ]:
# Output directory — all plots and tables land here
output_dir = os.path.join("..", "outputs")
os.makedirs(output_dir, exist_ok=True)

# Data directory — all data files land here
data_raw_dir = os.path.join("..", "data", "raw")
data_processed_dir = os.path.join("..", "data", "processed")

# Load Data

In [ ]:
# Load data
toaster_sentiment_df = pd.read_csv(os.path.join(data_processed_dir, "toaster_sentiment.csv"))

# Preview data
display(toaster_sentiment_df.head())

toaster_sentiment_df.info()

## Preprocess Data

In [ ]:
toaster_sentiment_df["RV_DT"] = pd.to_datetime(toaster_sentiment_df["RV_DT"], errors="coerce")

display(toaster_sentiment_df["RV_DT"].dt.to_period("Q"))

In [8]:
toaster_sentiment_df.columns

Index(['ASIN', 'P_TITLE', 'OP', 'DP', 'SP', 'FS', 'PRA_4.5', 'P_RTG',
       'RTG_P_NO', 'SELLER_LINK', 'IMAGE_URL', 'P_URL', 'RV_URL', 'PRFL_IMG',
       'PRFL_URL', 'RV_TTL', 'RVS', 'RVR', 'RSR', 'RVR_CONT', 'RV_DT', 'VP',
       'HLP_VT', 'IMG_PRST', 'TTL_RV', 'RVS_L', 'RV_TRANS', 'SUBJ', 'SRVS',
       'CP_RVS', 'TWRB_SENT', 'TWRB_SCORE', 'CHUNKED', 'SRB_SENT', 'SRB_SCORE',
       'RVRB_SENT', 'RVRB_SCORE'],
      dtype='str')

In [9]:
# Coerce Review_Date to datetime
toaster_sentiment_df["RV_DT"] = pd.to_datetime(toaster_sentiment_df["RV_DT"], errors="coerce")

# Coerce numeric columns to numeric
num_cols = [
    "OP", "DP", "SP", "FS", "PRA_4.5", "P_RTG", "RTG_P_NO",
    "RSR", "VP", "HLP_VT", "IMG_PRST", "TTL_RV", "RVS_L", "RV_TRANS",
    "SUBJ", "CP_RVS", "TWRB_SCORE", "SRB_SCORE", "RVRB_SCORE"]
    
for col in num_cols:
    toaster_sentiment_df[col] = pd.to_numeric(toaster_sentiment_df[col], errors="coerce") 

In [10]:
# Remove missing RVS / RSR values
toaster_sentiment_df = toaster_sentiment_df.dropna(subset=["RVS", "RSR"])

# Remove short reviews (less than 10 characters)
toaster_sentiment_df = toaster_sentiment_df[toaster_sentiment_df["RVS_L"] > 10].copy()

# Remove products with less than 3 reviews
toaster_sentiment_df = toaster_sentiment_df.groupby("ASIN").filter(lambda x: len(x) >= 3)

In [11]:
# Create quarter variables
toaster_sentiment_df["quarter"] = toaster_sentiment_df["RV_DT"].dt.to_period("Q")

## Summary of Key Numerical Variables

In [12]:

display(toaster_sentiment_df[["HLP_VT", "RVS_L", "TTL_RV", "SUBJ", "RSR", "CP_RVS", "TWRB_SCORE", "SRB_SCORE", "RVRB_SCORE"]].describe().round(3))

,HLP_VT,RVS_L,TTL_RV,SUBJ,RSR,CP_RVS,TWRB_SCORE,SRB_SCORE,RVRB_SCORE
count,13511.000,59602.000,59541.000,59598.000,59602.000,59598.000,59598.000,59598.000,59598.000
mean,4.305,188.372,2601.941,0.545,3.762,0.457,0.838,0.998,0.989
std,22.226,233.712,2084.289,0.256,1.588,0.458,0.160,0.013,0.039
min,1.000,11.000,1.000,0.000,1.000,-0.970,0.338,0.500,0.500
25%,1.000,54.000,716.000,0.421,2.000,0.060,0.747,0.999,0.996
50%,1.000,117.000,2220.000,0.588,5.000,0.625,0.906,0.999,0.998
75%,3.000,236.000,3977.000,0.717,5.000,0.836,0.968,0.999,0.999
max,1466.000,5979.000,14944.000,1.000,5.000,1.000,0.992,1.000,1.000


# Stage 1: Baseline Model

In [14]:
# Choose a sentiment variable from the multiple sentiment measures available.
sentiment_var = "RVRB_SENT"
toaster_sentiment_df["sentiment"] = toaster_sentiment_df[sentiment_var]

In [17]:
# B-spline basis with 4 degrees of freedom)
# RSR ~ B-spline(sentiment, df=4)
# Residual = distortion (rating minus sentiment-predicted rating)
spline_basis = dmatrix(
    "bs(sentiment, df=4, include_intercept=False)",
    data=toaster_sentiment_df, 
    return_type="dataframe"
)

TypeError: '<' not supported between instances of 'str' and 'float'